# ECGtizer V2 - Round-Trip Validation & Image-to-PTB-XL Pipeline

**Goal:** Validate that ECGtizer can correctly extract ECG signals from images and convert them into PTB-XL compatible format for arrhythmia classification.

**Pipeline:**
1. Load a PTB-XL record (ground truth) at 500Hz
2. Render it as a clinical-style ECG PDF using ECGtizer's own `Write_PDF`
3. Digitize it back with ECGtizer
4. Compare original vs. digitized signals (per-lead correlation)
5. Convert to PTB-XL format `(1000, 12)` and save as `.pt` tensor

**Key insight:** `ecg_plot` generates matplotlib charts that ECGtizer can't parse. We must use ECGtizer's own `Write_PDF` renderer to create proper clinical-style ECG PDFs with grid paper.

In [ ]:
# Cell 1: Setup & Imports
import sys
import os
import numpy as np
import pandas as pd
import ast
import wfdb
import torch
import matplotlib.pyplot as plt
import cv2
from scipy.signal import resample
from scipy.stats import pearsonr

# Fix import path: ensure ecgtizer package is found from the repo
ECGTIZER_REPO = os.path.join(os.getcwd(), 'ecgtizer')
if ECGTIZER_REPO not in sys.path:
    sys.path.insert(0, ECGTIZER_REPO)

from ecgtizer import ECGtizer
from ecgtizer.XML2PDF import Write_PDF # To make the PDFs for the test cases
from ecgtizer.PDF2XML import LEAD_TIME_3X4 # To get the lead time positions for the 3x4 classic layout (only for test case, Doctor uses 6X2 layout)
from ecgtizer.PDF2XML import LEAD_TIME_6X2 

print('All imports OK')
print(f'\nLead time positions (3x4 classic layout):')
for k, v in LEAD_TIME_6X2.items():
    if k != 'IIc':
        print(f'  {k}: samples {v[0]} to {v[1]} ({(v[1]-v[0])/500:.1f}s)')

In [ ]:
# Cell 2: Configuration
PTBXL_PATH = r'J:\Tesis\ptb-xl-1.0.3' #local path to PTB-XL dataset
TARGET_SAMPLING_RATE = 100  # Hz
TARGET_SAMPLES = 1000      # 10s * 100Hz (total)
OUTPUT_DIR = os.path.join(os.getcwd(), 'ecgtizer_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Lead name mappings
LEAD_NAMES_PTBXL = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF',
                     'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

LEAD_NAMES_ECGTIZER = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF',
                        'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

print(f'PTB-XL path: {PTBXL_PATH}')
print(f'Target: {TARGET_SAMPLES} samples @ {TARGET_SAMPLING_RATE}Hz')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# Cell 3: Load PTB-XL Dataset & Pick a Test Record
print('--- LOADING PTB-XL DATASET ---')

df_statements = pd.read_csv(os.path.join(PTBXL_PATH, 'scp_statements.csv'), index_col=0)
rhythm_codes = df_statements[df_statements.rhythm == 1].index.tolist()

ptbxl_df = pd.read_csv(os.path.join(PTBXL_PATH, 'ptbxl_database.csv'), index_col='ecg_id')
ptbxl_df['scp_codes_dict'] = ptbxl_df['scp_codes'].apply(ast.literal_eval)
ptbxl_df['rhythm_labels'] = ptbxl_df['scp_codes_dict'].apply(
    lambda y: list(set(y.keys()) & set(rhythm_codes))
)

ptbxl_clean = ptbxl_df[ptbxl_df['rhythm_labels'].apply(lambda x: len(x) > 0)].copy()

# Pick a type (in this case for testing we will pick records with 'SR' (sinus rhythm or normal) rhythm label)
sr_records = ptbxl_clean[ptbxl_clean['rhythm_labels'].apply(lambda x: 'SR' in x)]
test_record = sr_records.iloc[11] #can change the index to pick a different record for testing here

print(f'Found {len(ptbxl_clean)} records with rhythm labels')
print(f'SR records: {len(sr_records)}')
print(f'\nTest record:')
print(f'  HR path: {test_record["filename_hr"]}')
print(f'  LR path: {test_record["filename_lr"]}')
print(f'  Label:   {test_record["rhythm_labels"]}')

In [ ]:
# Cell 4: Load Ground Truth Signals

# Load 500Hz (for PDF generation, this is for ECGtizer to works at 500Hz internally)
signal_hr, meta_hr = wfdb.rdsamp(os.path.join(PTBXL_PATH, test_record['filename_hr']))
print(f'HR Signal: shape={signal_hr.shape}, fs={meta_hr["fs"]}Hz')
print(f'  Range: [{signal_hr.min():.4f}, {signal_hr.max():.4f}] mV')

# Load 100Hz (ground truth for our ML model which work on 100hz)
signal_lr, meta_lr = wfdb.rdsamp(os.path.join(PTBXL_PATH, test_record['filename_lr']))
print(f'\nLR Signal: shape={signal_lr.shape}, fs={meta_lr["fs"]}Hz')
print(f'  Range: [{signal_lr.min():.4f}, {signal_lr.max():.4f}] mV')
print(f'  Leads: {meta_lr["sig_name"]}')

---
## Part 1: Round-Trip Validation

**PTB-XL (digital)** -> **Clinical PDF** (via ECGtizer's Write_PDF) -> **ECGtizer digitization** -> **Compare**

This validates the entire pipeline with known ground truth. The idea is to first take the raw data, make it a PDF through ECGtizer built in xml to pdf function (this fix the problem of format not matching), then we passes it trough another function of ECGtizer to make it back again from PDF to xml and finally we compare the original (digital from PTB-Xl) and the one from ECGtizer.

In [ ]:
# Cell 5: Create Clinical ECG PDF (using ECGtizer's converter)

# ECGtizer's Write_PDF expects:
# - Dict with lead names as keys
# - Values in microvolts (uV) at 500Hz, 5000 samples per lead
# - Type1 = 3x4 layout (classic clinical format, can be changed to 6x2 layout if needed)

lead_order = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF',
              'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

ecg_dict = {}
for i, name in enumerate(lead_order):
    ecg_dict[name] = signal_hr[:, i] * 1000.0  # mV -> uV

pdf_path = os.path.join(OUTPUT_DIR, 'clinical_ecg_original_digitized.pdf')
Write_PDF(ecg_dict, pdf_path, type_of_pdf='type2', lead_IIc=ecg_dict['II'])

print(f'Clinical PDF saved: {pdf_path}')
print(f'Size: {os.path.getsize(pdf_path):,} bytes')

In [ ]:
# Cell 6: Digitize with ECGtizer
print('--- DIGITIZING WITH ECGTIZER ---')

ecg = ECGtizer(
    file=pdf_path, #path form above
    dpi=300, #Tested: 500 > 300 > 150 
    extraction_method='full',  # Tested: full > fragmented > lazy
    verbose=True,
)

print(f'\nDetected format: {ecg.TYPE}')
print(f'Extraction IsGood: {ecg.good}')

leads = ecg.extracted_lead
if isinstance(leads, dict):
    print(f'Leads extracted: {len(leads)}')
    print(f'Lead names: {list(leads.keys())}')
    
    print(f'\nPer-lead details:')
    for k, v in leads.items():
        v = np.array(v)
        nz = np.count_nonzero(v)
        print(f'  {k:>4s}: {len(v)} samples, {nz} non-zero, '
              f'range=[{v.min():.1f}, {v.max():.1f}] uV')

In [ ]:
# Cell 7: ECGtizer Built-in Plots per lead
if isinstance(leads, dict) and len(leads) > 0:
    print('Extracted leads plot:')
    ecg.plot()

In [ ]:
# Cell 7.1: ECGtizer Built-in Plot over (to see if it is aligned with the original image)
# This is important to check if the extracted leads are aligned with the original image, a visual way to know
if isinstance(leads, dict) and len(leads) > 0:
    print('\nOverlay on original image:')
    ecg.plot_over()

In [ ]:
# Cell 8: Convert ECGtizer Output to PTB-XL Format
# This one is important, this cell defines how to convert back to the raw data to later be made .pt for the actual moedl to run
#this is only for defining the function, the actual conversion is done in the next cell

def ecgtizer_to_ptbxl(extracted_leads, target_samples=1000):
    """
    Convert ECGtizer extracted leads to PTB-XL compatible numpy array.
    
    ECGtizer classic (3x4) output:
    - Each lead is 5000 samples long (10s at 500Hz)
    - Each lead's active data sits in a specific time window
      (e.g., Lead I = samples 0-1250, aVR = 1250-2500)
    - Rest is zero-padded
    - Values are in uV (scaled by AMPLITUDE_SCALE_UV=1000)
    
    PTB-XL format:
    - (1000, 12) array at 100Hz
    - Values in millivolts (mV)
    
    Returns:
        signal: numpy array of shape (target_samples, 12)
    """
    ecgtizer_names = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF',
                       'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
    
    signal = np.zeros((target_samples, 12))
    
    for col_idx, lead_name in enumerate(ecgtizer_names):
        if lead_name not in extracted_leads:
            print(f'  Warning: Lead {lead_name} not found')
            continue
        
        raw = np.array(extracted_leads[lead_name])
        t_start, t_end = LEAD_TIME_6X2[lead_name] 
        
        segment = raw[t_start:t_end]
        segment_mv = segment / 1000.0
        
        orig_start = t_start // 5
        orig_end = t_end // 5
        target_len = orig_end - orig_start
        
        if len(segment_mv) > 0:
            resampled = resample(segment_mv, target_len)
        else:
            resampled = np.zeros(target_len)
        
        # New Logic: Tiling with a Manual Phase Shift
        if len(resampled) > 0:
            tiled = np.zeros(target_samples)
            
            # 1. Place the first tile normally (0 to 5.0 seconds)
            tiled[0:target_len] = resampled
            
            # --- PHASE SHIFT SETTING ---
            shift_seconds = 0.3  # Push the second tile 0.25 seconds to the right
            shift_samples = int(shift_seconds * 100) # 100Hz = 50 samples
            
            # 2. Place the second tile shifted to the right
            start_idx = target_len + shift_samples
            if start_idx < target_samples:
                space_left = target_samples - start_idx
                tiled[start_idx:] = resampled[:space_left]
                
                # 3. Fill the 0.25s gap with a smooth flat baseline
                v_start = tiled[target_len - 1]
                v_end = tiled[start_idx]
                gap_size = (start_idx + 1) - (target_len - 1)
                tiled[target_len - 1 : start_idx + 1] = np.linspace(v_start, v_end, gap_size)
            
            signal[:, col_idx] = tiled
            
    return signal
print('Conversion function defined with 0.5s Phase Shift.')

In [ ]:
# Cell 9: Run Conversion & Compute Correlations

if isinstance(leads, dict) and len(leads) >= 12:
    # Convert
    signal_converted = ecgtizer_to_ptbxl(leads, TARGET_SAMPLES)
    
    # This 2 are for comparing to the original signal, to see if the conversion is correct or similar atleast
    print(f'Converted: shape={signal_converted.shape}, '
          f'range=[{signal_converted.min():.4f}, {signal_converted.max():.4f}] mV')
    
    print(f'Original:  shape={signal_lr.shape}, '
          f'range=[{signal_lr.min():.4f}, {signal_lr.max():.4f}] mV')
    
    # Per-lead correlation (on active segments only) this is for making the header of the "table"
    print(f'\n{"="*55}')
    print(f'{"Lead":>6s}  {"Time (s)":>10s}  {"Correlation":>12s}  {"Status":>8s}')
    print(f'{"="*55}')
    
    all_corrs = []
    for i, (name_ptb, name_ecg) in enumerate(zip(LEAD_NAMES_PTBXL, LEAD_NAMES_ECGTIZER)):
        t_start, t_end = LEAD_TIME_3X4[name_ecg]
        s, e = t_start // 5, t_end // 5
        
        orig_seg = signal_lr[s:e, i]
        conv_seg = signal_converted[s:e, i]
        
        if np.std(conv_seg) > 1e-6 and np.std(orig_seg) > 1e-6:
            corr, _ = pearsonr(orig_seg, conv_seg)
            all_corrs.append(corr)
            status = 'GOOD' if corr > 0.7 else 'FAIR' if corr > 0.4 else 'LOW'
            print(f'{name_ptb:>6s}  {s/100:.1f}-{e/100:.1f}s  {corr:>12.4f}  {status:>8s}')
    
    print(f'{"="*55}')
    print(f'{"AVG":>6s}  {"":>10s}  {np.mean(all_corrs):>12.4f}')
    print(f'{"MIN":>6s}  {"":>10s}  {np.min(all_corrs):>12.4f}')
    print(f'{"MAX":>6s}  {"":>10s}  {np.max(all_corrs):>12.4f}')
else:
    print(f'FAILED: Only {len(leads) if isinstance(leads, dict) else 0} leads extracted')

In [ ]:
# Cell 10: Visual Comparison (Active Segments)

if isinstance(leads, dict) and len(leads) >= 12:
    fig, axes = plt.subplots(6, 2, figsize=(18, 20))
    axes_flat = axes.flatten()
    
    for i, (name_ptb, name_ecg) in enumerate(zip(LEAD_NAMES_PTBXL, LEAD_NAMES_ECGTIZER)):
        ax = axes_flat[i]
        t_start, t_end = LEAD_TIME_3X4[name_ecg]
        s, e = t_start // 5, t_end // 5
        
        t_axis = np.linspace(s / 100.0, e / 100.0, e - s)
        orig_seg = signal_lr[s:e, i]
        conv_seg = signal_converted[s:e, i]
        
        ax.plot(t_axis, orig_seg, 'b-', alpha=0.7, linewidth=1.2, label='Original')
        ax.plot(t_axis, conv_seg, 'r-', alpha=0.7, linewidth=1.2, label='Digitized')
        
        if np.std(conv_seg) > 1e-6 and np.std(orig_seg) > 1e-6:
            corr, _ = pearsonr(orig_seg, conv_seg)
            ax.set_title(f'Lead {name_ptb} (r={corr:.3f})', fontsize=11)
        else:
            ax.set_title(f'Lead {name_ptb}', fontsize=11)
        
        ax.set_ylabel('mV')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Round-Trip Validation: PTB-XL -> Clinical PDF -> ECGtizer\n'
                 '(Active segments, aligned by time position)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    plot_path = os.path.join(OUTPUT_DIR, 'round_trip_comparison.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {plot_path}')

---
## Part 2: Save as .pt Tensor for ML Model

In [ ]:
# Cell 11: Save as .pt Tensor Files

def save_as_pt(signal, filepath, label=None):
    """Save signal as PyTorch tensor file."""
    tensor = torch.tensor(signal, dtype=torch.float32)

    if tensor.shape == (1000, 12):
        tensor = tensor.T
        
    torch.save(tensor, filepath)
    
    print(f'Saved: {filepath}')
    print(f'  Shape: {tensor.shape}, dtype: {tensor.dtype}')

if isinstance(leads, dict) and len(leads) >= 12:
    # Save digitized signal
    pt_digi = os.path.join(OUTPUT_DIR, 'digitized_ecg.pt')
    save_as_pt(signal_converted, pt_digi, label='SR')
    
    # Save original for comparison
    pt_orig = os.path.join(OUTPUT_DIR, 'original_ecg.pt')
    save_as_pt(signal_lr, pt_orig, label='SR')
    
    print(f'\nBoth files ready for ML model!')

In [ ]:
# Cell 12: Verify .pt Files
# Quick check to ensure the saved .pt files are correct and contain the expected data

for name in ['digitized_ecg.pt', 'original_ecg.pt']:
    fpath = os.path.join(OUTPUT_DIR, name)
    if os.path.exists(fpath):
        loaded = torch.load(fpath, weights_only=False)
        if isinstance(loaded, dict):
            t = loaded['signal']
            print(f'{name}:')
            print(f'  Shape: {t.shape}  (expected: [1000, 12])')
            print(f'  Dtype: {t.dtype}  (expected: float32)')
            print(f'  Label: {loaded["label"]}')
            print(f'  Range: [{t.min():.4f}, {t.max():.4f}] mV')
            print()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a massive plot for all 12 leads
fig, axes = plt.subplots(12, 1, figsize=(15, 24), sharex=True)
t_axis = np.linspace(0, 10, 1000)  # 10 seconds at 100Hz

for i, lead_name in enumerate(LEAD_NAMES_PTBXL):
    ax = axes[i]
    
    # Plot Original (Blue)
    ax.plot(t_axis, signal_lr[:, i], 'b-', alpha=0.7, linewidth=1.5, label='Original')
    
    # Plot Digitized (Red)
    ax.plot(t_axis, signal_converted[:, i], 'r--', alpha=0.8, linewidth=1.5, label='Digitized (Tiled)')
    
    ax.set_ylabel(f'{lead_name} (mV)', fontsize=12)
    ax.grid(True, alpha=0.4)
    
    # Add vertical lines to show where the tiles connect (every 2.5s)
    for jump in [2.5, 5.0, 7.5]:
        ax.axvline(x=jump, color='black', linestyle=':', alpha=0.5)

    if i == 0:
        ax.legend(loc='upper right', fontsize=12)

axes[-1].set_xlabel('Time (seconds)', fontsize=14)
plt.suptitle('Full 10-Second Comparison: Original vs Digitized', fontsize=18, y=0.91)
plt.show()

---
## Part 3: Test on Doctor's ECG Image

Test the pipeline on `normal.jpg` from the doctor folder.

**Note:** Low-resolution images (< 1500px wide) may not produce all 12 leads.
ECGtizer needs high-res images to distinguish individual ECG strips.

# Need to work on this
